# Aula 11 - Laboratório 4: Implementando Células Recorrentes com PyTorch

**Objetivo:** Implementar a mecânica interna de uma RNN simples estendendo `nn.Module`, demonstrar experimentalmente o problema do desaparecimento de gradientes e comparar a solução com as camadas `nn.LSTM` e `nn.GRU` do PyTorch.

---


In [ ]:
# Instalando as bibliotecas necessárias
!pip install -q datasets torch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.style.use('seaborn-v0_8-whitegrid')

## Parte 1: Revisão Teórica

Uma Rede Neural Recorrente (RNN) processa uma sequência desdobrando um grafo computacional no tempo. O estado oculto $h_t$ em cada passo é uma função da entrada atual $x_t$ e do estado oculto anterior $h_{t-1}$.

![RNN Unrolled](image_ab341a.jpg)

*Figura: Uma RNN simples desdobrada no tempo. Fonte: lxmls.herokuapp.com*

## Passo 2: Preparação dos Dados

Usaremos o dataset `rotten_tomatoes`. Vamos carregá-lo, criar um vocabulário manualmente e dividir o dataset em subconjuntos de frases 'curtas' e 'longas' para o nosso experimento.

In [ ]:
dataset = load_dataset("rotten_tomatoes")
# Tokenizador simples baseado em espaço
tokenizer = lambda text: text.lower().split()

# --- Construção Manual do Vocabulário ---
word_counts = Counter()
for item in dataset['train']:
    word_counts.update(tokenizer(item['text']))

vocab_list = sorted(word_counts, key=word_counts.get, reverse=True)
word_to_idx = {word: i+2 for i, word in enumerate(vocab_list)} # Começa do 2 para os tokens especiais
word_to_idx['<pad>'] = 0
word_to_idx['<unk>'] = 1
vocab_size = len(word_to_idx)
pad_idx = word_to_idx['<pad>']

text_pipeline = lambda x: [word_to_idx.get(token, word_to_idx['<unk>']) for token in tokenizer(x)]

# Separar os dados por comprimento de frase
train_data = list(dataset['train'])
short_sentences = [item for item in train_data if len(tokenizer(item['text'])) <= 5]
medium_sentences = [item for item in train_data if (len(tokenizer(item['text'])) > 10 and len(tokenizer(item['text'])) <= 30)]
long_sentences = [item for item in train_data if len(tokenizer(item['text'])) > 30]

class TextDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]['text'], self.data[idx]['label']

def collate_batch(batch):
    label_list, text_list = [], []
    for (_text, _label) in batch:
        label_list.append(_label)
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
        text_list.append(processed_text)
    return torch.tensor(label_list, dtype=torch.int64), nn.utils.rnn.pad_sequence(text_list, padding_value=pad_idx, batch_first=True)

short_loader = DataLoader(TextDataset(short_sentences), batch_size=32, shuffle=True, collate_fn=collate_batch)
medium_loader = DataLoader(TextDataset(medium_sentences), batch_size=32, shuffle=True, collate_fn=collate_batch)
long_loader = DataLoader(TextDataset(long_sentences), batch_size=32, shuffle=True, collate_fn=collate_batch)

In [ ]:
len(short_loader.dataset)

249

## Exercício 1: Implementando a RNN Simples (PyTorch)

Vamos começar implementando a mecânica de uma célula RNN e a camada que a itera no tempo.

In [ ]:
class SimpleRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        # TODO

    def forward(self, x_t, h_prev):
        # TODO

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn_cell = SimpleRNNCell(embed_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        h = torch.zeros(text.size(0), self.rnn_cell.hidden_size).to(device)

        for t in range(text.size(1)):
            h = self.rnn_cell(embedded[:, t, :], h)

        return self.fc(h)

## Exercício 2: Demonstrando a Limitação da RNN Simples

In [ ]:
def train(model, dataloader, optimizer, criterion):
    model.train()
    total_loss, total_acc = 0, 0
    for labels, text in dataloader:
        labels, text = labels.to(device), text.to(device)
        optimizer.zero_grad()
        output = model(text)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_acc += (output.argmax(1) == labels).sum().item()
    return total_acc / len(dataloader.dataset), total_loss / len(dataloader)

# TODO: Instancie e treine a SimpleRNN no `short_loader` e no `long_loader`.
# Plote as curvas de custo e acurácia. O que você observa?

## Exercício 3: Comparando com as Camadas Nativas do PyTorch

Agora, vamos usar as camadas `nn.LSTM` e `nn.GRU` do PyTorch, que são altamente otimizadas e resolvem o problema dos gradientes.

In [ ]:
class AdvancedRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, pad_idx, rnn_layer):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = rnn_layer(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        # A saída da RNN/LSTM/GRU é (output, hidden_state)
        # Para LSTM, hidden é uma tupla (h_n, c_n)
        # Para GRU e RNN, hidden é apenas h_n
        _, hidden = self.rnn(embedded)
        if isinstance(hidden, tuple):
            hidden = hidden[0]
        return self.fc(hidden.squeeze(0))

# TODO: Instancie e treine um modelo com suas implementacoes de LSTM, GRU, nn.LSTM e outro com nn.GRU
# no `long_loader`. Compare os resultados com a SimpleRNN.

## Exercício 4 (Bônus): RNN com Conexões Residuais

Inspirados nas ResNets, podemos criar uma célula RNN simples que usa ReLU e uma conexão residual.

In [ ]:
class ResRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        # TODO

    def forward(self, x_t, h_prev):
        # Equação da ResRNN: h_t = relu(W_xh*x_t + W_hh*h_{t-1} + h_{t-1})
        # TODO

# TODO (Avançado): Crie a classe wrapper, treine o modelo no `long_loader`
# e compare sua performance com os outros.